# 04 — Your First Neural Network

In this notebook, you build a small neural network with a custom `nn.Module`
and train it on synthetic tabular data.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1) Create a synthetic 3-class dataset

In [ ]:
def make_blob(center, n, scale=0.55):
    return torch.randn(n, 2) * scale + torch.tensor(center)

n_per_class = 400
X0 = make_blob(center=[-2.0, -1.5], n=n_per_class)
X1 = make_blob(center=[ 2.0, -1.0], n=n_per_class)
X2 = make_blob(center=[ 0.0,  2.0], n=n_per_class)

X = torch.cat([X0, X1, X2], dim=0)
y = torch.cat([
    torch.zeros(n_per_class),
    torch.ones(n_per_class),
    torch.full((n_per_class,), 2),
]).long()

dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

print("Dataset size:", len(dataset))

## 2) Define a custom model

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x):
        return self.net(x)

model = SimpleClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

print(model)

## 3) Train the model

In [ ]:
def evaluate(loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            preds = logits.argmax(dim=1)

            total_loss += loss.item() * yb.size(0)
            total_correct += (preds == yb).sum().item()
            total_samples += yb.size(0)

    return total_loss / total_samples, total_correct / total_samples

epochs = 25
for epoch in range(1, epochs + 1):
    model.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    if epoch == 1 or epoch % 5 == 0:
        train_loss, train_acc = evaluate(train_loader)
        val_loss, val_acc = evaluate(val_loader)
        print(
            f"Epoch {epoch:>2} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}"
        )

## 4) Make predictions

In [ ]:
model.eval()
sample_points = torch.tensor([
    [-2.1, -1.3],
    [ 2.2, -0.8],
    [ 0.1,  1.7],
    [ 0.0,  0.0],
], dtype=torch.float32).to(device)

with torch.no_grad():
    logits = model(sample_points)
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)

print("Predicted classes:", preds.cpu().tolist())
print("Probabilities:")
print(probs.cpu())

## 5) Exercises

1. Add `nn.Dropout(0.2)` after each ReLU. Compare validation accuracy.
2. Reduce hidden size from 32 to 8. What changes?
3. Save model weights with `torch.save(model.state_dict(), "first_nn.pt")`.

You now have enough PyTorch basics to move into time-series data preparation.